# Fatigue Prediction using Machine Learning

## Problem Statement
This project predicts fatigue based on user-reported symptoms using machine learning.

## Approach
- Data preprocessing and pivoting
- Conversion to classification problem
- Handling class imbalance using oversampling
- Training Random Forest model
- Threshold tuning

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import resample

# =========================
# 1. LOAD DATA
# =========================
DATA_PATH = "/kaggle/input/datasets/organizations/flaredown/flaredown-autoimmune-symptom-tracker/export.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
df = df.sample(100000, random_state=42)

# =========================
# 2. PREPROCESS
# =========================
symptoms = df[df['trackable_type'] == 'Symptom'].copy()

symptoms['trackable_value'] = pd.to_numeric(
    symptoms['trackable_value'], errors='coerce'
)
symptoms.dropna(subset=['trackable_value'], inplace=True)

pivot_df = symptoms.pivot_table(
    index=['user_id', 'checkin_date'],
    columns='trackable_name',
    values='trackable_value',
    aggfunc='mean'
).reset_index().fillna(0)

# Keep useful columns
non_zero_counts = (pivot_df != 0).sum()
selected_cols = non_zero_counts[non_zero_counts > 50].index.tolist()

if 'Fatigue' not in selected_cols:
    selected_cols.append('Fatigue')

clean_df = pivot_df[selected_cols]

# =========================
# 3. FEATURES + TARGET
# =========================
X = clean_df.select_dtypes(include=['number']).drop(columns=['Fatigue'])
y = (clean_df['Fatigue'] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 4. OVERSAMPLING
# =========================
train_df = pd.concat([X_train, y_train], axis=1)

majority = train_df[train_df['Fatigue'] == 0]
minority = train_df[train_df['Fatigue'] == 1]

print("Before balancing:", majority.shape, minority.shape)

minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

balanced_df = pd.concat([majority, minority_upsampled])
balanced_df = balanced_df.sample(frac=1, random_state=42)

X_train = balanced_df.drop(columns=['Fatigue'])
y_train = balanced_df['Fatigue']

print("After balancing:\n", y_train.value_counts())

# =========================
# 5. TRAIN MODEL
# =========================
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f"Model trained with {X.shape[1]} features.")

# =========================
# 6. EVALUATION (FINAL)
# =========================
y_probs = model.predict_proba(X_test)[:, 1]

threshold = 0.5  # 🔥 FINAL CHOICE
y_pred = (y_probs >= threshold).astype(int)

print("\nThreshold used:", threshold)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# =========================
# 7. PREDICTION FUNCTION
# =========================
def predict_fatigue(model, feature_columns, user_input, threshold=0.5):
    input_data = {col: 0 for col in feature_columns}
    input_data.update({k: v for k, v in user_input.items() if k in feature_columns})

    input_df = pd.DataFrame([input_data])

    prob = model.predict_proba(input_df)[0][1]
    pred = int(prob >= threshold)

    return pred, round(prob, 2)

# =========================
# 8. TEST
# =========================

# Real sample
sample_input = X_test.iloc[2].to_dict()
print("\nReal sample:", predict_fatigue(model, X.columns, sample_input))

# Custom input
current_symptoms = {
    'Headache': 3,
    'Nausea': 2,
    'Dizziness': 2,
    'Anxiety': 3,
    'Muscle Pain': 2,
    'Brain Fog': 3
}

print("Custom input:", predict_fatigue(model, X.columns, current_symptoms))

Before balancing: (31900, 84) (1000, 84)
After balancing:
 Fatigue
0    31900
1    31900
Name: count, dtype: int64
Model trained with 83 features.

Threshold used: 0.5
Accuracy: 0.36048632218844984

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.34      0.51      7975
           1       0.04      0.94      0.08       250

    accuracy                           0.36      8225
   macro avg       0.52      0.64      0.30      8225
weighted avg       0.97      0.36      0.50      8225


Confusion Matrix:
 [[2730 5245]
 [  15  235]]

Real sample: (1, np.float64(0.59))
Custom input: (0, np.float64(0.03))


## Insights

- The model achieves high recall (0.94), meaning it detects most fatigue cases
- Precision is low due to overlapping symptom patterns
- There is a clear trade-off between recall and precision
- Threshold tuning was used to balance model behavior

## Limitations

- Dataset is sparse and noisy
- Weak correlation between symptoms and fatigue
- High number of false positives

## Conclusion

This project demonstrates the challenges of predicting subjective health conditions using machine learning. While the model successfully detects fatigue cases, performance is limited by data quality and feature overlap.